# 02 Embedding And Store

这一部分开始对 `chunked_filings.jsonl` 做 embedding，并且采用“生成后立刻入库”的方式保存结果。

这里先使用本地 `SQLite` 作为第一版增量存储层，原因是：

- 非常适合 notebook 调试
- 可以随时中断
- 下次启动时可以直接从上次 `last_line` 继续
- 后面切换到 `Postgres + pgvector` 时，只需要替换存储层


In [1]:
from __future__ import annotations

import json
import os
import re
import sqlite3
from dataclasses import dataclass
from pathlib import Path
from typing import Optional

import numpy as np
from tqdm.auto import tqdm

BASE_DIR = Path('/Users/zhanghongyi/Desktop/26 Spring/Prof Zhao Finance Agent/Report_Crawer')
CHUNKED_JSONL = BASE_DIR / '04_Embedding' / 'chunked_data' / 'chunked_filings.jsonl'
STORE_DIR = BASE_DIR / '04_Embedding' / 'embedding_store'
STORE_DIR.mkdir(parents=True, exist_ok=True)

DB_PATH = STORE_DIR / 'filing_embeddings.sqlite3'
MODEL_NAME = os.getenv('REPORT_EMBED_MODEL', 'BAAI/bge-m3')
BATCH_SIZE = int(os.getenv('REPORT_EMBED_BATCH_SIZE', '16'))
PASSAGE_MAX_LENGTH = 512
SMOKE_TEST_CHUNKS = 8
FILTERED_BUILD_NAME = 'bge-m3-priority-sections-v1'
US_ALLOWED_SECTION_NAMES = {'item1', 'item1a', 'item1c', 'item3', 'item7'}
CN_ALLOWED_SECTION_MARKERS = (
    '第三章管理层讨论与分析',
    '第三节管理层讨论与分析',
    '第六章重要事项',
    '第六节重要事项',
)

print('Chunk input:', CHUNKED_JSONL)
print('Embedding db:', DB_PATH)
print('Model:', MODEL_NAME)
print('Batch size:', BATCH_SIZE)
print('Build name:', FILTERED_BUILD_NAME)


Chunk input: /Users/zhanghongyi/Desktop/26 Spring/Prof Zhao Finance Agent/Report_Crawer/04_Embedding/chunked_data/chunked_filings.jsonl
Embedding db: /Users/zhanghongyi/Desktop/26 Spring/Prof Zhao Finance Agent/Report_Crawer/04_Embedding/embedding_store/filing_embeddings.sqlite3
Model: BAAI/bge-m3
Batch size: 16
Build name: bge-m3-priority-sections-v1


## 1. 定义 chunk 数据结构

这一层只负责把 `chunked_filings.jsonl` 的每一行读成一个 `ChunkRecord`，后面的 embedder 和 store 都只面向这个统一对象。

In [2]:
@dataclass(slots=True)
class ChunkRecord:
    chunk_id: str
    market: str
    company_code: str
    ticker: Optional[str]
    company_name: str
    report_year: int
    filing_date: Optional[str]
    document_type: str
    title: str
    language: str
    source_path: str
    section_name: str
    chunk_index: int
    chunk_text: str
    token_count: int
    char_count: int


def load_chunk_record(line: str) -> ChunkRecord:
    return ChunkRecord(**json.loads(line))


## 2. 定义 embedding 模型

这里固定使用 `BAAI/bge-m3`。我们只取 dense embedding，因为后面要做的是 dense vector search。

设备选择顺序：

- 优先 `mps`
- 其次 `cuda`
- 最后回退 `cpu`


In [3]:
import torch
from transformers import AutoModel, AutoTokenizer


def pick_device() -> str:
    if getattr(torch.backends, 'mps', None) and torch.backends.mps.is_available():
        return 'mps'
    if torch.cuda.is_available():
        return 'cuda'
    return 'cpu'


class BGEPassageEmbedder:
    """A lightweight BGE-M3 dense embedder based on transformers."""

    def __init__(self, model_name: str = MODEL_NAME, batch_size: int = BATCH_SIZE):
        self.model_name = model_name
        self.batch_size = batch_size
        self.device = pick_device()
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.model = AutoModel.from_pretrained(model_name)
        self.model.to(self.device)
        self.model.eval()

    def embed_passages(self, texts: list[str]) -> np.ndarray:
        if not texts:
            return np.empty((0, 0), dtype=np.float32)

        outputs: list[np.ndarray] = []
        for start in range(0, len(texts), self.batch_size):
            batch_texts = texts[start:start + self.batch_size]
            encoded = self.tokenizer(
                batch_texts,
                padding=True,
                truncation=True,
                max_length=PASSAGE_MAX_LENGTH,
                return_tensors='pt',
            )
            encoded = {key: value.to(self.device) for key, value in encoded.items()}

            with torch.no_grad():
                model_output = self.model(**encoded)
                dense = model_output.last_hidden_state[:, 0]
                dense = torch.nn.functional.normalize(dense, p=2, dim=1)

            outputs.append(dense.detach().cpu().to(torch.float32).numpy())

        return np.concatenate(outputs, axis=0)


## 3. 定义本地增量存储层

这一版先使用 SQLite。每条 chunk embedding 保存时，会同时更新 `build_progress` 表里的 `last_line` 和 `processed_chunks`。

这样做的好处是：

- 中断后可以直接续跑
- 已经入库的不会丢
- 后面想做迁移时，也有非常清晰的中间结果


In [4]:
SQL_SCHEMA = '''
CREATE TABLE IF NOT EXISTS embeddings (
    chunk_id TEXT PRIMARY KEY,
    market TEXT NOT NULL,
    company_code TEXT NOT NULL,
    ticker TEXT,
    company_name TEXT NOT NULL,
    report_year INTEGER NOT NULL,
    filing_date TEXT,
    document_type TEXT NOT NULL,
    title TEXT NOT NULL,
    language TEXT NOT NULL,
    source_path TEXT NOT NULL,
    section_name TEXT NOT NULL,
    chunk_index INTEGER NOT NULL,
    token_count INTEGER NOT NULL,
    char_count INTEGER NOT NULL,
    chunk_text TEXT NOT NULL,
    embedding_blob BLOB NOT NULL,
    embedding_dim INTEGER NOT NULL,
    embedding_dtype TEXT NOT NULL,
    created_at TEXT NOT NULL DEFAULT CURRENT_TIMESTAMP
);

CREATE TABLE IF NOT EXISTS build_progress (
    build_name TEXT PRIMARY KEY,
    last_line INTEGER NOT NULL,
    processed_chunks INTEGER NOT NULL,
    updated_at TEXT NOT NULL DEFAULT CURRENT_TIMESTAMP
);

CREATE INDEX IF NOT EXISTS idx_embeddings_company_year
ON embeddings (market, company_code, report_year, document_type);
'''


class SQLiteEmbeddingStore:
    def __init__(self, db_path: Path = DB_PATH):
        self.db_path = db_path
        self.conn = sqlite3.connect(db_path)
        self.conn.execute('PRAGMA journal_mode=WAL')
        self.conn.execute('PRAGMA synchronous=NORMAL')

    def init_schema(self) -> None:
        self.conn.executescript(SQL_SCHEMA)
        self.conn.commit()

    def reset_all(self) -> None:
        with self.conn:
            self.conn.execute('DELETE FROM embeddings')
            self.conn.execute('DELETE FROM build_progress')

    def get_progress(self, build_name: str) -> tuple[int, int]:
        row = self.conn.execute(
            'SELECT last_line, processed_chunks FROM build_progress WHERE build_name = ?',
            (build_name,),
        ).fetchone()
        if row is None:
            return 0, 0
        return int(row[0]), int(row[1])

    def save_embedding_and_progress(
        self,
        build_name: str,
        line_number: int,
        processed_chunks: int,
        record: ChunkRecord,
        vector: np.ndarray,
    ) -> None:
        dense = np.asarray(vector, dtype=np.float32)
        with self.conn:
            self.conn.execute(
                '''
                INSERT OR REPLACE INTO embeddings (
                    chunk_id, market, company_code, ticker, company_name, report_year,
                    filing_date, document_type, title, language, source_path,
                    section_name, chunk_index, token_count, char_count, chunk_text,
                    embedding_blob, embedding_dim, embedding_dtype
                ) VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
                ''',
                (
                    record.chunk_id,
                    record.market,
                    record.company_code,
                    record.ticker,
                    record.company_name,
                    record.report_year,
                    record.filing_date,
                    record.document_type,
                    record.title,
                    record.language,
                    record.source_path,
                    record.section_name,
                    record.chunk_index,
                    record.token_count,
                    record.char_count,
                    record.chunk_text,
                    sqlite3.Binary(dense.tobytes()),
                    int(dense.shape[0]),
                    'float32',
                ),
            )
            self.conn.execute(
                '''
                INSERT INTO build_progress (build_name, last_line, processed_chunks, updated_at)
                VALUES (?, ?, ?, CURRENT_TIMESTAMP)
                ON CONFLICT(build_name) DO UPDATE SET
                    last_line = excluded.last_line,
                    processed_chunks = excluded.processed_chunks,
                    updated_at = CURRENT_TIMESTAMP
                ''',
                (build_name, line_number, processed_chunks),
            )

    def count_embeddings(self) -> int:
        row = self.conn.execute('SELECT COUNT(*) FROM embeddings').fetchone()
        return int(row[0])

    def close(self) -> None:
        self.conn.close()


## 4. 构建可续跑的 embedding 主流程

这一步是整个 notebook 的核心。

设计原则：

- 从 `chunked_filings.jsonl` 顺序读数据
- 一次取一小批做 embedding
- embedding 出来后逐条写入 SQLite
- 每写入一条，就更新一次 `build_progress`
- 下次重启时，直接从 `last_line` 继续


In [5]:
def normalize_section_name(section_name: str) -> str:
    return re.sub(r'\s+', '', (section_name or '')).lower()


def should_embed_record(record: ChunkRecord) -> bool:
    normalized = normalize_section_name(record.section_name)

    if record.market == 'US':
        return normalized in US_ALLOWED_SECTION_NAMES

    if record.market == 'CN':
        return any(marker in normalized for marker in CN_ALLOWED_SECTION_MARKERS)

    return False


def iter_chunk_records(start_line: int = 0):
    with CHUNKED_JSONL.open('r', encoding='utf-8', errors='replace') as f:
        for line_number, line in enumerate(f, start=1):
            if line_number <= start_line:
                continue
            line = line.strip()
            if not line:
                continue
            try:
                record = load_chunk_record(line)
            except json.JSONDecodeError:
                continue
            if should_embed_record(record):
                yield line_number, record


def embed_chunks_incrementally(
    build_name: str = FILTERED_BUILD_NAME,
    batch_size: int = BATCH_SIZE,
    max_items: Optional[int] = None,
):
    store = SQLiteEmbeddingStore(DB_PATH)
    store.init_schema()
    start_line, processed_total = store.get_progress(build_name)
    embedder = BGEPassageEmbedder(model_name=MODEL_NAME, batch_size=batch_size)

    pending_lines: list[int] = []
    pending_records: list[ChunkRecord] = []
    processed_this_run = 0

    def flush_pending() -> None:
        nonlocal processed_total, processed_this_run
        if not pending_records:
            return

        vectors = embedder.embed_passages([record.chunk_text for record in pending_records])
        for line_number, record, vector in zip(pending_lines, pending_records, vectors, strict=True):
            processed_total += 1
            processed_this_run += 1
            store.save_embedding_and_progress(
                build_name=build_name,
                line_number=line_number,
                processed_chunks=processed_total,
                record=record,
                vector=vector,
            )
            pbar.update(1)
            pbar.set_postfix(last_line=line_number, total_stored=processed_total)

        pending_lines.clear()
        pending_records.clear()

    try:
        print('Resume from line:', start_line)
        pbar = tqdm(total=max_items, desc='Embedding', unit='chunk') if max_items is not None else tqdm(desc='Embedding', unit='chunk')

        for line_number, record in iter_chunk_records(start_line=start_line):
            if max_items is not None and processed_this_run >= max_items:
                break

            pending_lines.append(line_number)
            pending_records.append(record)

            should_flush = len(pending_records) >= batch_size
            if max_items is not None and processed_this_run + len(pending_records) >= max_items:
                should_flush = True

            if should_flush:
                if max_items is not None and processed_this_run + len(pending_records) > max_items:
                    keep = max_items - processed_this_run
                    pending_lines[:] = pending_lines[:keep]
                    pending_records[:] = pending_records[:keep]
                flush_pending()

        flush_pending()
        pbar.close()

        stats = {
            'build_name': build_name,
            'resume_from_line': start_line,
            'processed_this_run': processed_this_run,
            'total_stored': store.count_embeddings(),
            'db_path': str(DB_PATH),
        }
        return stats
    finally:
        store.close()


## 5. 先检查当前数据库状态

如果是第一次跑，这里应该看到 `0 embeddings` 和 `0, 0` 的 progress。
如果之前已经跑过，那么这里会显示你可以从哪里继续。

In [6]:
store = SQLiteEmbeddingStore(DB_PATH)
store.init_schema()
print('Current embeddings:', store.count_embeddings())
print('Current progress:', store.get_progress(FILTERED_BUILD_NAME))
store.close()


Current embeddings: 7984
Current progress: (21260, 7984)


## 6. 先跑一个小样本 smoke test

第一次不要直接把 2.7 万个 chunk 全跑完。先跑 `8` 个，确认：

- 模型能正常加载
- embedding 能正常生成
- SQLite 能正常写入
- progress 能正常更新

如果这一步成功，下一格再把 `max_items` 去掉或调大。

In [7]:
smoke_stats = embed_chunks_incrementally(max_items=SMOKE_TEST_CHUNKS)
smoke_stats


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

Resume from line: 21260


Embedding:   0%|          | 0/8 [00:00<?, ?chunk/s]

{'build_name': 'bge-m3-priority-sections-v1',
 'resume_from_line': 21260,
 'processed_this_run': 8,
 'total_stored': 7992,
 'db_path': '/Users/zhanghongyi/Desktop/26 Spring/Prof Zhao Finance Agent/Report_Crawer/04_Embedding/embedding_store/filing_embeddings.sqlite3'}

## 7. 全量继续跑

确认 smoke test 没问题之后，再执行这一格。因为有 `build_progress`，所以你可以随时中断，下次直接重新运行这一格，它会从上次位置继续。


In [8]:
full_stats = embed_chunks_incrementally()
full_stats


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

Resume from line: 21268


Embedding: 0chunk [00:00, ?chunk/s]

KeyboardInterrupt: 